In [21]:
#  Imports and Setup
import pickle
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model

RANDOM_STATE = 42
tf.random.set_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

LATENT_DIM = 10
EPOCHS = 50
BATCH_SIZE = 16

In [22]:
#  Load Preprocessed Data
with open('../data/client_data_processed.pkl', 'rb') as f:
    client_data_processed = pickle.load(f)

print(f"Loaded {len(client_data_processed)} clients")
print(f"Latent dimension: {LATENT_DIM} (compression ratio: {15/LATENT_DIM:.1f}x)")

Loaded 4 clients
Latent dimension: 10 (compression ratio: 1.5x)


In [23]:
# Define Autoencoder Architecture
def build_autoencoder(input_dim, latent_dim):
    inputs = tf.keras.Input(shape=(input_dim,))
    
    # Encoder
    x = layers.Dense(32, activation='relu', 
                     kernel_regularizer=tf.keras.regularizers.l2(0.001))(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.2)(x)
    
    x = layers.Dense(20, activation='relu',
                     kernel_regularizer=tf.keras.regularizers.l2(0.001))(x)
    x = layers.BatchNormalization()(x)
    
    latent = layers.Dense(latent_dim, activation='linear', name='latent')(x)
    
    # Decoder
    x = layers.Dense(20, activation='relu',
                     kernel_regularizer=tf.keras.regularizers.l2(0.001))(latent)
    x = layers.BatchNormalization()(x)
    
    x = layers.Dense(32, activation='relu',
                     kernel_regularizer=tf.keras.regularizers.l2(0.001))(x)
    x = layers.BatchNormalization()(x)
    
    outputs = layers.Dense(input_dim, activation='linear')(x)
    
    autoencoder = Model(inputs, outputs, name='autoencoder')
    encoder = Model(inputs, latent, name='encoder')
    
    autoencoder.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='mse',
        metrics=['mae']
    )
    
    return autoencoder, encoder

In [24]:
# Train Autoencoders for Each Client
client_latent_features = []
client_encoders = []

for i, (x_c, y_c) in enumerate(client_data_processed):
    input_dim = x_c.shape[1]
    
    if x_c.shape[0] < 10:
        latent = x_c[:, :LATENT_DIM] if x_c.shape[1] >= LATENT_DIM else \
                 np.pad(x_c, ((0,0), (0, LATENT_DIM - x_c.shape[1])))
        client_latent_features.append(latent)
        client_encoders.append(None)
        continue
    
    autoencoder, encoder = build_autoencoder(input_dim, LATENT_DIM)
    
    early_stop = tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True
    )
    
    history = autoencoder.fit(
        x_c, x_c,
        epochs=EPOCHS,
        batch_size=min(BATCH_SIZE, max(1, x_c.shape[0])),
        validation_split=0.15,
        verbose=0,
        callbacks=[early_stop]
    )
    
    latent = encoder.predict(x_c, verbose=0)
    client_latent_features.append(latent)
    client_encoders.append(encoder)
    
    final_loss = history.history['val_loss'][-1] if 'val_loss' in history.history else history.history['loss'][-1]
    print(f"Client {i+1}: {x_c.shape} → {latent.shape} | Loss: {final_loss:.4f}")

Client 1: (1240, 15) → (1240, 10) | Loss: 0.0801
Client 2: (1240, 15) → (1240, 10) | Loss: 0.0791
Client 3: (1240, 15) → (1240, 10) | Loss: 0.0859
Client 4: (1240, 15) → (1240, 10) | Loss: 0.0802


In [25]:
# Save Results
all_latent = np.vstack(client_latent_features)
print(f"Combined latent shape: {all_latent.shape}")

with open('../data/client_latent_features.pkl', 'wb') as f:
    pickle.dump(client_latent_features, f)

with open('../data/client_encoders.pkl', 'wb') as f:
    pickle.dump(client_encoders, f)

print("Saved latent features and encoders")

Combined latent shape: (4960, 10)
Saved latent features and encoders
